#  Analisis Keanekaragaman Herpetofauna TNBB
## Tutorial Sederhana: scikit-bio

**Sumber data:** Amarasinghe et al. (2021). *Herpetofaunal diversity of West Bali National Park, Indonesia.*  
Global Ecology and Conservation, 28, e01638.

**Tujuan tutorial ini:**  
Memahami cara menghitung indeks keanekaragaman (Shannon, Simpson, Evenness) menggunakan Python, lalu membuat grafik sederhana.

---

##  1. Teori Dasar — Apa itu Indeks Keanekaragaman?

Ketika kita mempelajari suatu komunitas hewan di lapangan, kita ingin tahu:
- Ada berapa **jenis** (spesies) yang hidup di sana?
- Seberapa **merata** jumlah individu antar spesies?
- Seberapa **beragam** komunitas tersebut secara keseluruhan?

Untuk menjawab pertanyaan itu, ekologi menggunakan beberapa **indeks keanekaragaman**:

---

###  Shannon-Wiener Index (H')

**Shannon menghitung apa?**  
Mengukur **keberagaman** suatu komunitas. Semakin banyak spesies DAN semakin merata jumlah individu tiap spesies → nilai H' makin tinggi.

**Rumus:**
$$H' = -\sum_{i=1}^{S} p_i \times \ln(p_i)$$

Di mana:
- $S$ = jumlah spesies
- $p_i$ = proporsi individu spesies ke-$i$ (jumlah spesies i ÷ total individu)
- $\ln$ = logaritma natural

**Kriteria nilai H':**
| Nilai H' | Keterangan |
|----------|------------|
| H' < 1.0 | Keanekaragaman **rendah** |
| 1.0 – 3.0 | Keanekaragaman **sedang** |
| H' > 3.0 | Keanekaragaman **tinggi** |

---

###  Simpson's Dominance Index (D)

**Simpson menghitung apa?**  
Mengukur **dominasi** — seberapa besar komunitas dikuasai oleh satu atau sedikit spesies.

**Rumus:**
$$D = \sum_{i=1}^{S} p_i^2$$

**Cara membaca nilai D:**
- D mendekati **1.0** → komunitas **didominasi** satu spesies (keanekaragaman rendah)
- D mendekati **0.0** → komunitas **merata** (keanekaragaman tinggi)

> ️ **Catatan scikit-bio:** Fungsi `simpson()` di scikit-bio mengembalikan **Gini-Simpson = 1 − D**, bukan D langsung.  
> Untuk mendapat D = Σpi², gunakan: `D = 1 - simpson(counts)`

---

###  Pielou's Evenness (J)

**Evenness menghitung apa?**  
Mengukur **kemerataan** distribusi individu antar spesies. Apakah semua spesies punya jumlah individu yang mirip, atau ada yang jauh lebih banyak?

**Rumus:**
$$J = \frac{H'}{\ln(S)}$$

- J berkisar antara **0 sampai 1**
- J = 1 → semua spesies punya jumlah individu yang **sama persis** (sangat merata)
- J mendekati 0 → ada satu spesies yang **jauh mendominasi** jumlahnya

---

##  2. Instalasi dan Import Library

In [ ]:
# Jalankan cell ini di Google Colab
!pip install scikit-bio -q
print('Instalasi selesai!')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import fungsi alpha diversity dari scikit-bio
from skbio.diversity.alpha import (
    shannon,            # Indeks Shannon-Wiener
    simpson,            # Indeks Gini-Simpson
    observed_features,  # Kekayaan spesies (S)
    pielou_e            # Kemerataan Pielou (J)
)

print(' Semua library berhasil diimport!')

##  3. Memuat Dataset

In [ ]:
# Mount Google Drive terlebih dahulu
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Sesuaikan path file CSV di Google Drive kamu
jalur_file = '/content/drive/My Drive/tnbb_herpetofauna.csv'

# Baca file CSV, kolom 'species' dijadikan index baris
df = pd.read_csv(jalur_file, index_col='species')

print('Dataset berhasil dimuat!')
print(f'Ukuran data: {df.shape[0]} spesies x {df.shape[1]} habitat')
print()
df

In [ ]:
# Ringkasan singkat: berapa spesies dan individu di tiap habitat?
daftar_habitat = df.columns.tolist()

print('Ringkasan data per habitat:')
print(f'{"Habitat":<25} {"Spesies":>8} {"Individu":>10}')
print('-' * 45)
for h in daftar_habitat:
    jml_spesies  = (df[h] > 0).sum()
    jml_individu = df[h].sum()
    print(f'{h:<25} {jml_spesies:>8} {jml_individu:>10}')


##  4. Pengenalan Fungsi scikit-bio

Semua fungsi alpha diversity ada di modul `skbio.diversity.alpha`.

| Fungsi | Menghitung | Output |
|--------|-----------|--------|
| `shannon(counts, base=np.e)` | Shannon-Wiener H' | angka desimal |
| `simpson(counts)` | Gini-Simpson **(1 − Σpi²)** | angka desimal |
| `observed_features(counts)` | Jumlah spesies yang hadir | bilangan bulat |
| `pielou_e(counts)` | Kemerataan Pielou's J | angka desimal |

**Catatan penting:** Semua fungsi menerima input berupa **array jumlah individu** (bukan proporsi), contoh: `[30, 8, 4, 9, ...]`

### 4.1 `shannon()` — Shannon-Wiener Index

**Contoh sederhana sebelum pakai data asli:**

Bayangkan dua komunitas:
- Komunitas A: 3 spesies, masing-masing 10 individu → **sangat merata**
- Komunitas B: 3 spesies, 28 / 1 / 1 individu → **didominasi satu spesies**

H' komunitas A pasti lebih tinggi dari komunitas B.

In [ ]:
# ---------- Contoh sederhana dulu ----------
komunitas_merata  = np.array([10, 10, 10])   # sangat merata
komunitas_dominan = np.array([28, 1, 1])     # didominasi satu spesies

H_merata  = shannon(komunitas_merata,  base=np.e)
H_dominan = shannon(komunitas_dominan, base=np.e)

print('=== Contoh Sederhana shannon() ===')
print(f"Komunitas merata  : H' = {H_merata:.4f}")
print(f"Komunitas dominan : H' = {H_dominan:.4f}")
print()
print("Komunitas merata lebih beragam (H' lebih tinggi)")


In [ ]:
# ---------- Hitung untuk data TNBB ----------
ind_lembab = df['hutan_lembab'].values.astype(int)

H_lembab = shannon(ind_lembab, base=np.e)

print('=== shannon() untuk Hutan Lembab TNBB ===')
print(f"H' (Shannon) = {H_lembab:.4f}")
print()

# Interpretasi otomatis
if H_lembab < 1.0:
    kategori = 'RENDAH'
elif H_lembab <= 3.0:
    kategori = 'SEDANG'
else:
    kategori = 'TINGGI'

print(f"Interpretasi: Keanekaragaman {kategori} (H' = {H_lembab:.2f})")


### 4.2 `simpson()` — Simpson's Dominance Index

In [ ]:
# simpson() di scikit-bio mengembalikan Gini-Simpson (1 - D)
# Untuk mendapat D (Dominance): D = 1 - simpson()

gini_lembab = simpson(ind_lembab)
D_lembab    = 1 - gini_lembab

print('=== simpson() untuk Hutan Lembab TNBB ===')
print(f'Output scikit-bio  (Gini-Simpson) = {gini_lembab:.4f}')
print(f"Setelah konversi   (Simpson's D)  = {D_lembab:.4f}")
print()
print('Interpretasi:')
print(f'  D = {D_lembab:.4f} -> mendekati 0, komunitas MERATA')


### 4.3 `observed_features()` — Kekayaan Spesies (S)

In [ ]:
S_lembab = observed_features(ind_lembab)

print('=== observed_features() untuk Hutan Lembab TNBB ===')
print(f'Jumlah spesies (S) = {int(S_lembab)} spesies')


### 4.4 `pielou_e()` — Kemerataan (Evenness)

In [ ]:
J_lembab = pielou_e(ind_lembab)

print('=== pielou_e() untuk Hutan Lembab TNBB ===')
print(f"Pielou's J = {J_lembab:.4f}")
print()
print('Interpretasi:')
print(f'  J = {J_lembab:.2f} -> mendekati 1.0, individu terdistribusi cukup merata')


##  5. Hitung Semua Indeks untuk Semua Habitat

In [ ]:
# Loop semua habitat, hitung semua indeks, simpan ke tabel
hasil = []

for habitat in daftar_habitat:
    ind = df[habitat].values.astype(int)

    S = int(observed_features(ind))
    H = round(shannon(ind, base=np.e), 4)
    D = round(1 - simpson(ind), 4)
    J = round(pielou_e(ind), 4)

    hasil.append({
        'Habitat'      : habitat.replace('_', ' ').title(),
        'S (Spesies)'  : S,
        "H' (Shannon)" : H,
        'D (Simpson)'  : D,
        'J (Evenness)' : J
    })

tabel_hasil = pd.DataFrame(hasil)

print('Hasil Analisis Alpha Diversity Herpetofauna TNBB')
print('=' * 60)
print(tabel_hasil.to_string(index=False))
print()
print("Keterangan:")
print("  S  = jumlah spesies   |  H' = Shannon (makin tinggi = makin beragam)")
print("  D  = Simpson dominance (mendekati 0 = merata, mendekati 1 = didominasi)")
print("  J  = Evenness Pielou  (mendekati 1 = individu terdistribusi merata)")


##  6. Visualisasi — Grafik Shannon-Wiener

In [ ]:
# Ambil data dari tabel hasil
nama_habitat = [r['Habitat'] for r in hasil]
nilai_H      = [r["H' (Shannon)"] for r in hasil]

warna = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12']

plt.figure(figsize=(8, 5))

batang = plt.bar(nama_habitat, nilai_H, color=warna, width=0.5)

plt.axhline(y=1.0, color='gray',  linestyle='--', linewidth=1, label="Batas rendah (H'=1.0)")
plt.axhline(y=3.0, color='black', linestyle='--', linewidth=1, label="Batas tinggi (H'=3.0)")

for batang_i, nilai in zip(batang, nilai_H):
    plt.text(
        batang_i.get_x() + batang_i.get_width() / 2,
        nilai + 0.05,
        f'{nilai:.2f}',
        ha='center', va='bottom',
        fontsize=11, fontweight='bold'
    )

plt.title("Shannon-Wiener Index (H')\nKeanekaragaman Herpetofauna TNBB", fontsize=13)
plt.ylabel("Indeks Shannon-Wiener (H')", fontsize=11)
plt.xlabel('Tipe Habitat', fontsize=11)
plt.ylim(0, 3.2)
plt.legend(fontsize=9)
plt.tight_layout()

plt.savefig('shannon_tnbb.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik tersimpan sebagai shannon_tnbb.png')


##  7. Interpretasi Hasil

Berdasarkan grafik dan tabel di atas, kita dapat menyimpulkan:

**Shannon-Wiener (H'):**
- **Hutan Lembab** memiliki H' tertinggi → keanekaragaman **sedang-tinggi**, spesies tersebar merata
- **Hutan Gugur** dan **Savana** memiliki nilai serupa → keanekaragaman **sedang**
- **Perkebunan Jati** memiliki H' sangat rendah → keanekaragaman **rendah**, komunitas didominasi satu spesies (*Eutropis rugifera*)

**Mengapa Perkebunan Jati paling rendah?**  
Perkebunan jati yang ditinggalkan menyediakan sedikit sumber daya habitat (tumbuhan bawah, air, tempat berlindung), sehingga hanya spesies generalis yang mampu bertahan.

---

**Referensi:** Amarasinghe et al. (2021). *Global Ecology and Conservation*, 28, e01638.  
https://doi.org/10.1016/j.gecco.2021.e01638